In [1]:
# Inputs: Wild_Occurrence_Urban_Intersect , Urban Areas

# Outputs: Wild_Occurrence_Line_detail 

# Purpose: Create the sequenced lines between each fire occurrence point and each urban area

# Notable functions:# find all fires within 100km of fire occurrence; calculate distance and direction; 
                    # draw line between them; break up line into points; keep only wildfires; 
                    # get occurrence_burn_area output for GEE

In [1]:
import pandas as pd 
pd.set_option('display.max_columns', None)
import pyproj
pyproj.network.set_network_enabled(False)
from pyproj import Proj, Transformer
from pyproj import Geod
import geopandas

In [2]:
def convert_to_epsg(df, df2, epsg_int):
    df = df.to_crs(f"EPSG:{epsg_int}")
    df2.to_crs(df.crs, inplace=True)
    return df, df2

def create_points(row, geometry, point_separation):
    import shapely
    import numpy as np
    geom = row[geometry]
    if geom is None or geom.is_empty:
        return []
    point_list = [geom.interpolate(distance=x) for x in np.arange(start=0, stop=geom.length, step=point_separation)]
    return point_list
    
    

In [3]:
def find_nearby_fires(fires_dataframe, urban_file, buffer_size):
    u_df = geopandas.read_file(urban_file)
    f_df, u_df = convert_to_epsg(fires_dataframe, u_df, 5070)
    
    u_short = u_df[['geometry']]
    u_short["UrbanGeom"] = u_short["geometry"]
    
    f_df = f_df[f_df["FIRE_TYPE"] == 'Wildfire']
    f_df["fires_df_buffer"] = f_df.buffer(buffer_size)
    
    f_short = f_df[["FIRE_ID", "fires_df_buffer"]]
    f_short.rename(columns={"fires_df_buffer": "geometry"}, inplace=True)
    
    fires_2 = geopandas.sjoin(f_short, u_short, how='left')
    
    f_urb = pd.merge(f_df, fires_2, on="FIRE_ID", how='left')
    
    f_urb = f_urb.reset_index(drop=True)
    f_urb["Total_Distance_To_Urban"] = f_urb["geometry_x"].distance(f_urb["UrbanGeom"])
    
    f_urb = f_urb[f_urb['Total_Distance_To_Urban'].notna()]
    
    return f_urb
   
    
    

In [4]:
occurrences_df = geopandas.read_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartOneOutput.shp")


occurrences_df.head()

,FIRE_ID,FIRE_NAME,FIRE_TYPE,IG_DATE,LATITUDE,LONGITUDE,CrossedWUI,UrbanOccur,geometry
0,TX3632910272820250318,HIGH LONESOME,Wildfire,2025-03-18,36.41493694055366,-102.5994235231072,0.0,0.0,POINT (-102.59942 36.41494)
1,NM3586210475920250314,MOGOTE HILL,Wildfire,2025-03-14,35.850345790841196,-104.6948825531825,0.0,0.0,POINT (-104.69488 35.85035)
2,SD4465410198420250310,ROUTE 13,Wildfire,2025-03-10,44.6870693101975,-101.84227869317823,0.0,0.0,POINT (-101.84228 44.68707)
3,NE4185910104020250225,DISMAL RIVER RANCH,Wildfire,2025-02-25,41.78861628081379,-100.98318408926657,0.0,0.0,POINT (-100.98318 41.78862)
4,TX3604510084520250314,WINDMILL,Wildfire,2025-03-14,36.06960022791254,-100.7319597261261,0.0,0.0,POINT (-100.73196 36.0696)


In [64]:
occ_2020 = occurrences_df[occurrences_df["IG_DATE"] > "2019-12-31"]
#occ_2010 = occurrences_df[(occurrences_df["IG_DATE"] > "2009-12-31") & (occurrences_df["IG_DATE"] <= "2019-12-31")]
#occ_2000 = occurrences_df[occurrences_df["IG_DATE"] <= "2009-12-31"]


In [5]:
#urban_2000 = r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\WUI\Urban Areas\tl_2008_us_uac00.shp"
#urban_2010 = r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\WUI\Urban Areas\tl_2010_us_uac10.shp"
urban_2020 = r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\WUI\Urban Areas\tl_2020_us_uac20.shp"

In [65]:
#occ_2000 = find_nearby_fires(occ_2000, urban_2000, 10000)
#occ_2010 = find_nearby_fires(occ_2010, urban_2010, 10000)

# this actually removes the one WUI breach from the dataset - it travels over 15km to do so!
occ_2020 = find_nearby_fires(occ_2020, urban_2020, 10000)

In [66]:
occ_test = occ_2020[['FIRE_ID','CrossedWUI']]
occ_test.groupby(['CrossedWUI']).size()

CrossedWUI
0.0    7
dtype: int64

In [67]:
fires_df_urb =  occ_2020 #pd.concat([occ_2000, occ_2010, occ_2020])

In [68]:
fires_df_urb.head()

,FIRE_ID,FIRE_NAME,FIRE_TYPE,IG_DATE,LATITUDE,LONGITUDE,CrossedWUI,UrbanOccur,geometry_x,fires_df_buffer,geometry_y,index_right,UrbanGeom,Total_Distance_To_Urban
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276
43,TX2902809527920250730,SALT LAKE,Wildfire,2025-07-30,29.031814300574997,-95.2859149491568,0.0,0.0,POINT (69631.248 662315.605),"POLYGON ((79631.248 662315.605, 79583.096 6613...","POLYGON ((79631.248 662315.605, 79583.096 6613...",558.0,"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",7447.011886
56,ND4787810329020250327,32ND,Wildfire,2025-03-27,47.91127441689051,-103.3209964653087,0.0,0.0,POINT (-551076.926 2789613.286),"POLYGON ((-541076.926 2789613.286, -541125.079...","POLYGON ((-541076.926 2789613.286, -541125.079...",1838.0,"MULTIPOLYGON (((-549005.425 2779255.475, -5489...",9059.720809
57,MN4455409582920250322,VALLERS,Wildfire,2025-03-22,44.55933112379632,-95.8359081346575,0.0,0.0,POINT (13005.822 2396670.432),"POLYGON ((23005.822 2396670.432, 22957.669 239...","POLYGON ((23005.822 2396670.432, 22957.669 239...",2057.0,"MULTIPOLYGON (((18588.604 2383238.729, 18590.2...",9532.197200
60,VA3712108003520250129,CAHAS MOUNTTAIN FIRE,Wildfire,2025-01-29,37.11740658490369,-80.0455726696443,0.0,0.0,POINT (1397755.248 1681517.591),"POLYGON ((1407755.248 1681517.591, 1407707.095...","POLYGON ((1407755.248 1681517.591, 1407707.095...",2632.0,"MULTIPOLYGON (((1403771.933 1693101.998, 14037...",8866.644834


In [69]:
len(fires_df_urb)

7

In [70]:
transformer = Transformer.from_crs("EPSG:5070", "EPSG:4326", always_xy=True)
lon1, lat1 = transformer.transform(fires_df_urb["geometry_x"].x, fires_df_urb["geometry_x"].y)
centroids = fires_df_urb["UrbanGeom"].centroid
lon2, lat2 = transformer.transform(centroids.x, centroids.y)

geod = Geod(ellps="WGS84")
from_angle, to_angle, distance = geod.inv(lon1, lat1, lon2, lat2)
fires_df_urb["Direction_To_Urban"] = from_angle
fires_df_urb["Direction_From_Urban_To_Fire"] = to_angle
fires_df_urb["Distance_To_Urban"] = distance

angles = [i for i in range(-1, 360, 20)]

fires_df_urb["Angles"] = fires_df_urb["Direction_To_Urban"].astype(float).apply(lambda x: [((x + a + 180) % 360) - 180 for a in angles])

fires_df_urb.head()


,FIRE_ID,FIRE_NAME,FIRE_TYPE,IG_DATE,LATITUDE,LONGITUDE,CrossedWUI,UrbanOccur,geometry_x,fires_df_buffer,geometry_y,index_right,UrbanGeom,Total_Distance_To_Urban,Direction_To_Urban,Direction_From_Urban_To_Fire,Distance_To_Urban,Angles
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,"[-163.64711543514673, -143.64711543514673, -12..."
43,TX2902809527920250730,SALT LAKE,Wildfire,2025-07-30,29.031814300574997,-95.2859149491568,0.0,0.0,POINT (69631.248 662315.605),"POLYGON ((79631.248 662315.605, 79583.096 6613...","POLYGON ((79631.248 662315.605, 79583.096 6613...",558.0,"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",7447.011886,-102.417782,77.526018,11557.440586,"[-103.41778189936902, -83.41778189936902, -63...."
56,ND4787810329020250327,32ND,Wildfire,2025-03-27,47.91127441689051,-103.3209964653087,0.0,0.0,POINT (-551076.926 2789613.286),"POLYGON ((-541076.926 2789613.286, -541125.079...","POLYGON ((-541076.926 2789613.286, -541125.079...",1838.0,"MULTIPOLYGON (((-549005.425 2779255.475, -5489...",9059.720809,159.205336,-20.749402,12880.076726,"[158.20533636600157, 178.20533636600157, -161...."
57,MN4455409582920250322,VALLERS,Wildfire,2025-03-22,44.55933112379632,-95.8359081346575,0.0,0.0,POINT (13005.822 2396670.432),"POLYGON ((23005.822 2396670.432, 22957.669 239...","POLYGON ((23005.822 2396670.432, 22957.669 239...",2057.0,"MULTIPOLYGON (((18588.604 2383238.729, 18590.2...",9532.197200,163.008902,-16.957880,12908.886738,"[162.00890160684554, -177.99109839315446, -157..."
60,VA3712108003520250129,CAHAS MOUNTTAIN FIRE,Wildfire,2025-01-29,37.11740658490369,-80.0455726696443,0.0,0.0,POINT (1397755.248 1681517.591),"POLYGON ((1407755.248 1681517.591, 1407707.095...","POLYGON ((1407755.248 1681517.591, 1407707.095...",2632.0,"MULTIPOLYGON (((1403771.933 1693101.998, 14037...",8866.644834,19.205724,-160.746827,21148.945457,"[18.205724227060756, 38.205724227060756, 58.20..."


In [71]:
fires_df_urb = fires_df_urb.explode("Angles")
fires_df_urb["Angles"] = fires_df_urb["Angles"].astype(float)
fires_df_urb["Dummy_Distance"] = 10000
fires_df_urb.head()

,FIRE_ID,FIRE_NAME,FIRE_TYPE,IG_DATE,LATITUDE,LONGITUDE,CrossedWUI,UrbanOccur,geometry_x,fires_df_buffer,geometry_y,index_right,UrbanGeom,Total_Distance_To_Urban,Direction_To_Urban,Direction_From_Urban_To_Fire,Distance_To_Urban,Angles,Dummy_Distance
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-163.647115,10000
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-143.647115,10000
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-123.647115,10000
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-103.647115,10000
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-83.647115,10000


In [72]:
lon3, lat3 = transformer.transform(fires_df_urb["geometry_x"].x, fires_df_urb["geometry_x"].y)
angles_column = fires_df_urb["Angles"]
distance_column = fires_df_urb["Dummy_Distance"]
print(len(lon3), len(lat3), len(angles_column))

133 133 133


In [73]:
lon4, lat4, _ = geod.fwd(lon3, lat3, angles_column, distance_column)
fires_df_urb["ConstructPoint_Lon"] = lon4
fires_df_urb["ConstructPoint_Lat"] = lat4

fires_df_urb.head()

,FIRE_ID,FIRE_NAME,FIRE_TYPE,IG_DATE,LATITUDE,LONGITUDE,CrossedWUI,UrbanOccur,geometry_x,fires_df_buffer,geometry_y,index_right,UrbanGeom,Total_Distance_To_Urban,Direction_To_Urban,Direction_From_Urban_To_Fire,Distance_To_Urban,Angles,Dummy_Distance,ConstructPoint_Lon,ConstructPoint_Lat
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-163.647115,10000,-120.115571,36.149258
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-143.647115,10000,-120.150164,36.163137
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-123.647115,10000,-120.176831,36.185767
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-103.647115,10000,-120.192357,36.214425
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-83.647115,10000,-120.194858,36.245657


In [74]:
fires_df_urb["TestPoint"] = geopandas.points_from_xy(fires_df_urb["ConstructPoint_Lon"].astype(float),fires_df_urb["ConstructPoint_Lat"].astype(float), crs=5070)
fires_df_urb = geopandas.GeoDataFrame(fires_df_urb, geometry="TestPoint", crs=4326)
fires_df_urb["TestPoint"] = fires_df_urb["TestPoint"].to_crs(5070)
fires_df_urb.head()

,FIRE_ID,FIRE_NAME,FIRE_TYPE,IG_DATE,LATITUDE,LONGITUDE,CrossedWUI,UrbanOccur,geometry_x,fires_df_buffer,geometry_y,index_right,UrbanGeom,Total_Distance_To_Urban,Direction_To_Urban,Direction_From_Urban_To_Fire,Distance_To_Urban,Angles,Dummy_Distance,ConstructPoint_Lon,ConstructPoint_Lat,TestPoint
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-163.647115,10000,-120.115571,36.149258,POINT (-2127223.864 1726812.053)
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-143.647115,10000,-120.150164,36.163137,POINT (-2129818.543 1729091.418)
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-123.647115,10000,-120.176831,36.185767,POINT (-2131481.59 1732142.322)
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-103.647115,10000,-120.192357,36.214425,POINT (-2132012.416 1735596.809)
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-83.647115,10000,-120.194858,36.245657,POINT (-2131347.026 1739038.244)


In [75]:
fires_df_urb["IgnitionUrbanLine"] = fires_df_urb['geometry_x'].shortest_line(fires_df_urb["TestPoint"])
fires_df_urb.head()

,FIRE_ID,FIRE_NAME,FIRE_TYPE,IG_DATE,LATITUDE,LONGITUDE,CrossedWUI,UrbanOccur,geometry_x,fires_df_buffer,geometry_y,index_right,UrbanGeom,Total_Distance_To_Urban,Direction_To_Urban,Direction_From_Urban_To_Fire,Distance_To_Urban,Angles,Dummy_Distance,ConstructPoint_Lon,ConstructPoint_Lat,TestPoint,IgnitionUrbanLine
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-163.647115,10000,-120.115571,36.149258,POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223..."
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-143.647115,10000,-120.150164,36.163137,POINT (-2129818.543 1729091.418),"LINESTRING (-2122095.131 1735488.284, -2129818..."
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-123.647115,10000,-120.176831,36.185767,POINT (-2131481.59 1732142.322),"LINESTRING (-2122095.131 1735488.284, -2131481..."
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-103.647115,10000,-120.192357,36.214425,POINT (-2132012.416 1735596.809),"LINESTRING (-2122095.131 1735488.284, -2132012..."
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-83.647115,10000,-120.194858,36.245657,POINT (-2131347.026 1739038.244),"LINESTRING (-2122095.131 1735488.284, -2131347..."


In [76]:
fires_df_urb = fires_df_urb[fires_df_urb["UrbanGeom"].intersects(fires_df_urb["IgnitionUrbanLine"])]
fires_df_urb.drop_duplicates(subset=["FIRE_ID", "ConstructPoint_Lon", "ConstructPoint_Lat"], inplace=True)
fires_df_urb.head()

,FIRE_ID,FIRE_NAME,FIRE_TYPE,IG_DATE,LATITUDE,LONGITUDE,CrossedWUI,UrbanOccur,geometry_x,fires_df_buffer,geometry_y,index_right,UrbanGeom,Total_Distance_To_Urban,Direction_To_Urban,Direction_From_Urban_To_Fire,Distance_To_Urban,Angles,Dummy_Distance,ConstructPoint_Lon,ConstructPoint_Lat,TestPoint,IgnitionUrbanLine
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-163.647115,10000,-120.115571,36.149258,POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223..."
31,CA3622612010420250902,MARMON,Wildfire,2025-09-02,36.23573658949745,-120.08428531234935,0.0,0.0,POINT (-2122095.131 1735488.284),"POLYGON ((-2112095.131 1735488.284, -2112143.2...","POLYGON ((-2112095.131 1735488.284, -2112143.2...",1300.0,"POLYGON ((-2124521.091 1731501.212, -2124849.2...",2723.555276,-162.647115,17.345449,3794.215812,-143.647115,10000,-120.150164,36.163137,POINT (-2129818.543 1729091.418),"LINESTRING (-2122095.131 1735488.284, -2129818..."
43,TX2902809527920250730,SALT LAKE,Wildfire,2025-07-30,29.031814300574997,-95.2859149491568,0.0,0.0,POINT (69631.248 662315.605),"POLYGON ((79631.248 662315.605, 79583.096 6613...","POLYGON ((79631.248 662315.605, 79583.096 6613...",558.0,"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",7447.011886,-102.417782,77.526018,11557.440586,-103.417782,10000,-95.385752,29.010841,POINT (59911.137 659925.518),"LINESTRING (69631.248 662315.605, 59911.137 65..."
43,TX2902809527920250730,SALT LAKE,Wildfire,2025-07-30,29.031814300574997,-95.2859149491568,0.0,0.0,POINT (69631.248 662315.605),"POLYGON ((79631.248 662315.605, 79583.096 6613...","POLYGON ((79631.248 662315.605, 79583.096 6613...",558.0,"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",7447.011886,-102.417782,77.526018,11557.440586,-83.417782,10000,-95.387908,29.042118,POINT (59678.566 663386.991),"LINESTRING (69631.248 662315.605, 59678.566 66..."
43,TX2902809527920250730,SALT LAKE,Wildfire,2025-07-30,29.031814300574997,-95.2859149491568,0.0,0.0,POINT (69631.248 662315.605),"POLYGON ((79631.248 662315.605, 79583.096 6613...","POLYGON ((79631.248 662315.605, 79583.096 6613...",558.0,"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",7447.011886,-102.417782,77.526018,11557.440586,-163.417782,10000,-95.315189,28.945340,POINT (66845.761 652721.462),"LINESTRING (69631.248 662315.605, 66845.761 65..."


In [77]:
fires_df_urb = fires_df_urb[['FIRE_ID', 'FIRE_TYPE', 'IG_DATE', 'Total_Distance_To_Urban', 'Distance_To_Urban', 'Angles', 'geometry_x', 'UrbanGeom', 'TestPoint', 'IgnitionUrbanLine', 'CrossedWUI']].reset_index()
fires_df_urb.rename(columns={"geometry_x": "geometry", "TestPoint": "LineEndPoint"}, inplace=True)
fires_df_urb.head(100)

,index,FIRE_ID,FIRE_TYPE,IG_DATE,Total_Distance_To_Urban,Distance_To_Urban,Angles,geometry,UrbanGeom,LineEndPoint,IgnitionUrbanLine,CrossedWUI
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0
1,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-143.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2129818.543 1729091.418),"LINESTRING (-2122095.131 1735488.284, -2129818...",0.0
2,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-103.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (59911.137 659925.518),"LINESTRING (69631.248 662315.605, 59911.137 65...",0.0
3,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-83.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (59678.566 663386.991),"LINESTRING (69631.248 662315.605, 59678.566 66...",0.0
4,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-163.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (66845.761 652721.462),"LINESTRING (69631.248 662315.605, 66845.761 65...",0.0
5,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-143.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (63724.623 654250.504),"LINESTRING (69631.248 662315.605, 63724.623 65...",0.0
6,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-123.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (61316.259 656752.448),"LINESTRING (69631.248 662315.605, 61316.259 65...",0.0
7,56,ND4787810329020250327,Wildfire,2025-03-27,9059.720809,12880.076726,158.205336,POINT (-551076.926 2789613.286),"MULTIPOLYGON (((-549005.425 2779255.475, -5489...",POINT (-548055.947 2780139.096),"LINESTRING (-551076.926 2789613.286, -548055.9...",0.0
8,60,VA3712108003520250129,Wildfire,2025-01-29,8866.644834,21148.945457,18.205724,POINT (1397755.248 1681517.591),"MULTIPOLYGON (((1403771.933 1693101.998, 14037...",POINT (1399203.32 1691491.274),"LINESTRING (1397755.248 1681517.591, 1399203.3...",0.0
9,60,VA3712108003520250129,Wildfire,2025-01-29,8866.644834,21148.945457,-1.794276,POINT (1397755.248 1681517.591),"MULTIPOLYGON (((1403771.933 1693101.998, 14037...",POINT (1395763.183 1691415.895),"LINESTRING (1397755.248 1681517.591, 1395763.1...",0.0


In [78]:
burn_df = geopandas.read_file(r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\Validation\mtbs_perimeter_data\mtbs_perims_DD.shp")
burn_df.crs

<Geographic 2D CRS: EPSG:4269>
Name: NAD83
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Scotia; Nunavut; Ontario; Prince Edward Island; Quebec; Saskatchewan; Yukon. Puerto Rico. United States (USA) - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Hawaii; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming. US Virgin Islands. British Virgin Islands

In [79]:
#fires_df_urb.set_geometry('')
burn_df = burn_df.to_crs(5070)
burn_df.rename(columns={'event_id': 'FIRE_ID'}, inplace=True)

In [80]:
fires_df_urb = pd.merge(fires_df_urb, burn_df[['FIRE_ID', 'geometry']], on='FIRE_ID')
fires_df_urb.rename(columns={'geometry_y': 'Burn_Area'}, inplace=True)
fires_df_urb.head()

,index,FIRE_ID,FIRE_TYPE,IG_DATE,Total_Distance_To_Urban,Distance_To_Urban,Angles,geometry_x,UrbanGeom,LineEndPoint,IgnitionUrbanLine,CrossedWUI,Burn_Area
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9..."
1,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-143.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2129818.543 1729091.418),"LINESTRING (-2122095.131 1735488.284, -2129818...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9..."
2,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-103.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (59911.137 659925.518),"LINESTRING (69631.248 662315.605, 59911.137 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622..."
3,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-83.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (59678.566 663386.991),"LINESTRING (69631.248 662315.605, 59678.566 66...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622..."
4,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-163.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (66845.761 652721.462),"LINESTRING (69631.248 662315.605, 66845.761 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622..."


In [81]:
fires_df_urb["OccurrenceUrbanID"] = fires_df_urb.index + 1
fires_df_urb.head(100)

,index,FIRE_ID,FIRE_TYPE,IG_DATE,Total_Distance_To_Urban,Distance_To_Urban,Angles,geometry_x,UrbanGeom,LineEndPoint,IgnitionUrbanLine,CrossedWUI,Burn_Area,OccurrenceUrbanID
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1
1,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-143.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2129818.543 1729091.418),"LINESTRING (-2122095.131 1735488.284, -2129818...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",2
2,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-103.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (59911.137 659925.518),"LINESTRING (69631.248 662315.605, 59911.137 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",3
3,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-83.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (59678.566 663386.991),"LINESTRING (69631.248 662315.605, 59678.566 66...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",4
4,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-163.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (66845.761 652721.462),"LINESTRING (69631.248 662315.605, 66845.761 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",5
5,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-143.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (63724.623 654250.504),"LINESTRING (69631.248 662315.605, 63724.623 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",6
6,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-123.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (61316.259 656752.448),"LINESTRING (69631.248 662315.605, 61316.259 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",7
7,56,ND4787810329020250327,Wildfire,2025-03-27,9059.720809,12880.076726,158.205336,POINT (-551076.926 2789613.286),"MULTIPOLYGON (((-549005.425 2779255.475, -5489...",POINT (-548055.947 2780139.096),"LINESTRING (-551076.926 2789613.286, -548055.9...",0.0,"POLYGON ((-551090.672 2788746.726, -551150.912...",8
8,60,VA3712108003520250129,Wildfire,2025-01-29,8866.644834,21148.945457,18.205724,POINT (1397755.248 1681517.591),"MULTIPOLYGON (((1403771.933 1693101.998, 14037...",POINT (1399203.32 1691491.274),"LINESTRING (1397755.248 1681517.591, 1399203.3...",0.0,"POLYGON ((1396979.011 1682427.285, 1396981.246...",9
9,60,VA3712108003520250129,Wildfire,2025-01-29,8866.644834,21148.945457,-1.794276,POINT (1397755.248 1681517.591),"MULTIPOLYGON (((1403771.933 1693101.998, 14037...",POINT (1395763.183 1691415.895),"LINESTRING (1397755.248 1681517.591, 1395763.1...",0.0,"POLYGON ((1396979.011 1682427.285, 1396981.246...",10


In [82]:
len(fires_df_urb)

13

In [83]:
fires_df_urb["point_list"] = fires_df_urb.apply(lambda x: create_points(row=x, geometry="IgnitionUrbanLine", point_separation=50), axis=1)
fires_df_urb.head()

,index,FIRE_ID,FIRE_TYPE,IG_DATE,Total_Distance_To_Urban,Distance_To_Urban,Angles,geometry_x,UrbanGeom,LineEndPoint,IgnitionUrbanLine,CrossedWUI,Burn_Area,OccurrenceUrbanID,point_list
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,[POINT (-2122095.1309723007 1735488.2836294293...
1,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-143.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2129818.543 1729091.418),"LINESTRING (-2122095.131 1735488.284, -2129818...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",2,[POINT (-2122095.1309723007 1735488.2836294293...
2,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-103.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (59911.137 659925.518),"LINESTRING (69631.248 662315.605, 59911.137 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",3,"[POINT (69631.24827974396 662315.6047843664), ..."
3,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-83.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (59678.566 663386.991),"LINESTRING (69631.248 662315.605, 59678.566 66...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",4,"[POINT (69631.24827974396 662315.6047843664), ..."
4,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-163.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (66845.761 652721.462),"LINESTRING (69631.248 662315.605, 66845.761 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",5,"[POINT (69631.24827974396 662315.6047843664), ..."


In [84]:
df_points = fires_df_urb.explode(column="point_list")
df_points['point_order'] = df_points.groupby(['FIRE_ID', 'OccurrenceUrbanID']).cumcount()
df_points.head()

,index,FIRE_ID,FIRE_TYPE,IG_DATE,Total_Distance_To_Urban,Distance_To_Urban,Angles,geometry_x,UrbanGeom,LineEndPoint,IgnitionUrbanLine,CrossedWUI,Burn_Area,OccurrenceUrbanID,point_list,point_order
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122095.1309723007 1735488.2836294293),0
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122120.574313459 1735445.2413665857),1
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122146.0176546173 1735402.1991037421),2
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122171.460995776 1735359.1568408986),3
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122196.9043369344 1735316.114578055),4


In [85]:
df_points.groupby(["OccurrenceUrbanID"]).count()

,index,FIRE_ID,FIRE_TYPE,IG_DATE,Total_Distance_To_Urban,Distance_To_Urban,Angles,geometry_x,UrbanGeom,LineEndPoint,IgnitionUrbanLine,CrossedWUI,Burn_Area,point_list,point_order
OccurrenceUrbanID,,,,,,,,,,,,,,,
1,202,202,202,202,202,202,202,202,202,202,202,202,202,202,202
2,201,201,201,201,201,201,201,201,201,201,201,201,201,201,201
3,201,201,201,201,201,201,201,201,201,201,201,201,201,201,201
4,201,201,201,201,201,201,201,201,201,201,201,201,201,201,201
5,200,200,200,200,200,200,200,200,200,200,200,200,200,200,200
6,200,200,200,200,200,200,200,200,200,200,200,200,200,200,200
7,201,201,201,201,201,201,201,201,201,201,201,201,201,201,201
8,199,199,199,199,199,199,199,199,199,199,199,199,199,199,199
9,202,202,202,202,202,202,202,202,202,202,202,202,202,202,202


In [86]:
df_points

,index,FIRE_ID,FIRE_TYPE,IG_DATE,Total_Distance_To_Urban,Distance_To_Urban,Angles,geometry_x,UrbanGeom,LineEndPoint,IgnitionUrbanLine,CrossedWUI,Burn_Area,OccurrenceUrbanID,point_list,point_order
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122095.1309723007 1735488.2836294293),0
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122120.574313459 1735445.2413665857),1
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122146.0176546173 1735402.1991037421),2
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122171.460995776 1735359.1568408986),3
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122196.9043369344 1735316.114578055),4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12,67,NC3522508227420250302,Wildfire,2025-03-02,3371.016984,8490.584685,99.169519,POINT (1232904.772 1441338.006),"MULTIPOLYGON (((1245513.809 1436337.414, 12455...",POINT (1242821.309 1441154.338),"LINESTRING (1232904.772 1441338.006, 1242821.3...",0.0,"POLYGON ((1233890.327 1442141.841, 1233937.36 ...",13,POINT (1242603.1091186872 1441158.3789666267),194
12,67,NC3522508227420250302,Wildfire,2025-03-02,3371.016984,8490.584685,99.169519,POINT (1232904.772 1441338.006),"MULTIPOLYGON (((1245513.809 1436337.414, 12455...",POINT (1242821.309 1441154.338),"LINESTRING (1232904.772 1441338.006, 1242821.3...",0.0,"POLYGON ((1233890.327 1442141.841, 1233937.36 ...",13,POINT (1242653.1005447707 1441157.4530518658),195
12,67,NC3522508227420250302,Wildfire,2025-03-02,3371.016984,8490.584685,99.169519,POINT (1232904.772 1441338.006),"MULTIPOLYGON (((1245513.809 1436337.414, 12455...",POINT (1242821.309 1441154.338),"LINESTRING (1232904.772 1441338.006, 1242821.3...",0.0,"POLYGON ((1233890.327 1442141.841, 1233937.36 ...",13,POINT (1242703.091970854 1441156.527137105),196
12,67,NC3522508227420250302,Wildfire,2025-03-02,3371.016984,8490.584685,99.169519,POINT (1232904.772 1441338.006),"MULTIPOLYGON (((1245513.809 1436337.414, 12455...",POINT (1242821.309 1441154.338),"LINESTRING (1232904.772 1441338.006, 1242821.3...",0.0,"POLYGON ((1233890.327 1442141.841, 1233937.36 ...",13,POINT (1242753.0833969375 1441155.601222344),197


In [87]:
df_points["Y"] = df_points.apply(lambda x: 1 if pd.notnull(x['point_list']) and x['point_list'].intersects(x["Burn_Area"]) else 0, axis=1)

In [88]:
df_points["IsUrban"] = df_points.apply(lambda x: 1 if pd.notnull(x['point_list']) and x['point_list'].intersects(x["UrbanGeom"]) else 0, axis=1)

In [89]:
df_points["WUIBreach"] = df_points.apply(lambda x: 1 if x["Y"] == 1 and x["IsUrban"] == 1 else 0, axis=1)


In [90]:
df_points["Y"].value_counts()

Y
0    2339
1     271
Name: count, dtype: int64

In [91]:
df_points["IsUrban"].value_counts()

IsUrban
0    2378
1     232
Name: count, dtype: int64

In [92]:
df_points["WUIBreach"].value_counts()

WUIBreach
0    2610
Name: count, dtype: int64

In [93]:
last_urban_point = (df_points[df_points["IsUrban"] == 1].groupby("OccurrenceUrbanID")["point_order"].max())

In [94]:
df_points["last_urban_point"] = df_points["OccurrenceUrbanID"].map(last_urban_point)

In [95]:
df_points["RemoveFlag"] = (df_points["point_order"] > df_points["last_urban_point"]).astype(int)

In [96]:
df_points = df_points[df_points["RemoveFlag"] == 0]

In [97]:
df_points["Y"].value_counts()

Y
0    2116
1     271
Name: count, dtype: int64

In [98]:
df_points.head(1000)

,index,FIRE_ID,FIRE_TYPE,IG_DATE,Total_Distance_To_Urban,Distance_To_Urban,Angles,geometry_x,UrbanGeom,LineEndPoint,IgnitionUrbanLine,CrossedWUI,Burn_Area,OccurrenceUrbanID,point_list,point_order,Y,IsUrban,WUIBreach,last_urban_point,RemoveFlag
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122095.1309723007 1735488.2836294293),0,1,0,0,92.0,0
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122120.574313459 1735445.2413665857),1,1,0,0,92.0,0
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122146.0176546173 1735402.1991037421),2,1,0,0,92.0,0
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122171.460995776 1735359.1568408986),3,1,0,0,92.0,0
0,31,CA3622612010420250902,Wildfire,2025-09-02,2723.555276,3794.215812,-163.647115,POINT (-2122095.131 1735488.284),"POLYGON ((-2124521.091 1731501.212, -2124849.2...",POINT (-2127223.864 1726812.053),"LINESTRING (-2122095.131 1735488.284, -2127223...",0.0,"POLYGON ((-2122511.636 1732917.819, -2122510.9...",1,POINT (-2122196.9043369344 1735316.114578055),4,1,0,0,92.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-143.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (63724.623 654250.504),"LINESTRING (69631.248 662315.605, 63724.623 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",6,POINT (66706.50442496271 658322.0633254707),99,0,0,0,199.0,0
5,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-143.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (63724.623 654250.504),"LINESTRING (69631.248 662315.605, 63724.623 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",6,POINT (66676.96155774269 658281.7245228556),100,0,0,0,199.0,0
5,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-143.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (63724.623 654250.504),"LINESTRING (69631.248 662315.605, 63724.623 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",6,POINT (66647.41869052268 658241.3857202404),101,0,0,0,199.0,0
5,43,TX2902809527920250730,Wildfire,2025-07-30,7447.011886,11557.440586,-143.417782,POINT (69631.248 662315.605),"MULTIPOLYGON (((58546.23 662919.738, 58557.093...",POINT (63724.623 654250.504),"LINESTRING (69631.248 662315.605, 63724.623 65...",0.0,"POLYGON ((70426.575 662539.276, 70431.994 6622...",6,POINT (66617.87582330266 658201.0469176254),102,0,0,0,199.0,0


In [99]:
df_out = df_points.copy()

df_out['geometry'] = df_out['point_list']
df_out = geopandas.GeoDataFrame(df_out, geometry=df_out['point_list'], crs="EPSG:5070")

df_out = df_out[["FIRE_ID", "IG_DATE", "FIRE_TYPE", "OccurrenceUrbanID", "point_order", "geometry"]]

In [100]:
df_out

,FIRE_ID,IG_DATE,FIRE_TYPE,OccurrenceUrbanID,point_order,geometry
0,CA3622612010420250902,2025-09-02,Wildfire,1,0,POINT (-2122095.131 1735488.284)
0,CA3622612010420250902,2025-09-02,Wildfire,1,1,POINT (-2122120.574 1735445.241)
0,CA3622612010420250902,2025-09-02,Wildfire,1,2,POINT (-2122146.018 1735402.199)
0,CA3622612010420250902,2025-09-02,Wildfire,1,3,POINT (-2122171.461 1735359.157)
0,CA3622612010420250902,2025-09-02,Wildfire,1,4,POINT (-2122196.904 1735316.115)
...,...,...,...,...,...,...
12,NC3522508227420250302,2025-03-02,Wildfire,13,95,POINT (1237653.958 1441250.045)
12,NC3522508227420250302,2025-03-02,Wildfire,13,96,POINT (1237703.949 1441249.119)
12,NC3522508227420250302,2025-03-02,Wildfire,13,97,POINT (1237753.941 1441248.193)
12,NC3522508227420250302,2025-03-02,Wildfire,13,98,POINT (1237803.932 1441247.267)


In [101]:
df_points.rename(columns={"point_list": "geometry", "Angles": "UrbanAngle", "NAME20": "City"}, inplace=True)
df_points.drop(columns=['geometry_x', 'UrbanGeom', 'LineEndPoint', 'Total_Distance_To_Urban', 'Distance_To_Urban', 'IgnitionUrbanLine', 'CrossedWUI', 'Burn_Area', 'RemoveFlag'], inplace=True)
df_points.head()

,index,FIRE_ID,FIRE_TYPE,IG_DATE,UrbanAngle,OccurrenceUrbanID,geometry,point_order,Y,IsUrban,WUIBreach,last_urban_point
0,31,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,POINT (-2122095.131 1735488.284),0,1,0,0,92.0
0,31,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,POINT (-2122120.574 1735445.241),1,1,0,0,92.0
0,31,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,POINT (-2122146.018 1735402.199),2,1,0,0,92.0
0,31,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,POINT (-2122171.461 1735359.157),3,1,0,0,92.0
0,31,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,POINT (-2122196.904 1735316.115),4,1,0,0,92.0


In [102]:
df_points.drop(columns=["index"], inplace=True)
df_points = geopandas.GeoDataFrame(df_points, geometry=df_points['geometry'], crs="EPSG:5070")

In [103]:
df_out.to_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartTwoOutput3.shp")

C:\Users\jezkn\AppData\Local\Temp\ipykernel_31912\3527808094.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  df_out.to_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartTwoOutput3.shp")
C:\Users\jezkn\Anaconda3\envs\geo_env2\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'OccurrenceUrbanID' to 'Occurrence'
  ogr_write(
C:\Users\jezkn\Anaconda3\envs\geo_env2\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'point_order' to 'point_orde'
  ogr_write(


In [104]:
df_points.to_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartTwoOutputShape3.shp")

C:\Users\jezkn\AppData\Local\Temp\ipykernel_31912\2708785527.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  df_points.to_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartTwoOutputShape3.shp")
C:\Users\jezkn\Anaconda3\envs\geo_env2\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'OccurrenceUrbanID' to 'Occurrence'
  ogr_write(
C:\Users\jezkn\Anaconda3\envs\geo_env2\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'point_order' to 'point_orde'
  ogr_write(
C:\Users\jezkn\Anaconda3\envs\geo_env2\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'last_urban_point' to 'last_urban'
  ogr_write(
